# 16 · Degraded Training (Data Augmentation)
Augment the UrbanSound8K training set with degraded audio features (low-pass 4 kHz, white noise, mild clipping) to train mixed models (SVM-mixed, RF-mixed), evaluate robustness improvements on held-out test Fold 10, and export new candidate artifacts.

In [1]:
# Cell 1 – interpreter setup & imports
import sys
from pathlib import Path
import time
import hashlib

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib
from tqdm import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from src.preprocessing.audio import load_fixed, extract_features, SR
from src.preprocessing.degrade import degrade

print('Successfully imported preprocessing, models, and metrics')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported preprocessing, models, and metrics


In [2]:
# Cell 2 – verify sha256 of inputs, load train & test data
def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

labels_path = REPO_ROOT / 'data/processed/labels.csv'
features_path = REPO_ROOT / 'data/processed/audio_features.npy'
split_path = REPO_ROOT / 'data/processed/split_indices.npz'
degraded_eval_path = REPO_ROOT / 'reports/degraded_eval.csv'

labels_sha = compute_sha256(labels_path)
features_sha = compute_sha256(features_path)
split_sha = compute_sha256(split_path)
eval_sha = compute_sha256(degraded_eval_path)

print(f'labels.csv sha256        : {labels_sha}')
print(f'audio_features.npy sha256: {features_sha}')
print(f'split_indices.npz sha256 : {split_sha}')
print(f'degraded_eval.csv sha256 : {eval_sha}')

EXPECTED_LABELS_SHA = '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854'
EXPECTED_FEAT_SHA   = 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a'
EXPECTED_SPLIT_SHA  = '2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b'
EXPECTED_EVAL_SHA   = 'a9eba6ddf21622f860cbba0d1b2d2c7599d5b94d943c7e111c434bd5e368721e'

assert labels_sha == EXPECTED_LABELS_SHA, f'labels.csv SHA mismatch: {labels_sha}'
assert features_sha == EXPECTED_FEAT_SHA, f'audio_features.npy SHA mismatch: {features_sha}'
assert split_sha == EXPECTED_SPLIT_SHA, f'split_indices.npz SHA mismatch: {split_sha}'
assert eval_sha == EXPECTED_EVAL_SHA, f'degraded_eval.csv SHA mismatch: {eval_sha}'

labels = pd.read_csv(labels_path)
audio_features = np.load(features_path)

with np.load(split_path) as split_data:
    train_idx = split_data['train_idx']
    test_idx = split_data['test_idx']

X_train_clean = audio_features[train_idx]
y_train = labels.loc[train_idx, 'app_class'].to_numpy()
X_test_clean = audio_features[test_idx]
y_test = labels.loc[test_idx, 'app_class'].to_numpy()

print(f'Loaded train rows: {len(train_idx)}, test rows: {len(test_idx)}')
print(f'X_train_clean shape: {X_train_clean.shape}, y_train shape: {y_train.shape}')
print(f'X_test_clean shape : {X_test_clean.shape}, y_test shape : {y_test.shape}')

assert X_train_clean.shape == (5753, 128), f'Expected X_train (5753, 128), got {X_train_clean.shape}'
assert X_test_clean.shape == (605, 128), f'Expected X_test (605, 128), got {X_test_clean.shape}'
assert len(y_train) == 5753, f'Expected 5753 in y_train, got {len(y_train)}'
assert len(y_test) == 605, f'Expected 605 in y_test, got {len(y_test)}'

print('Cell 2 SHA256 asserts and loads passed ✓')

labels.csv sha256        : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
audio_features.npy sha256: ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
split_indices.npz sha256 : 2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b
degraded_eval.csv sha256 : a9eba6ddf21622f860cbba0d1b2d2c7599d5b94d943c7e111c434bd5e368721e
Loaded train rows: 5753, test rows: 605
X_train_clean shape: (5753, 128), y_train shape: (5753,)
X_test_clean shape : (605, 128), y_test shape : (605,)
Cell 2 SHA256 asserts and loads passed ✓


In [3]:
# Cell 3 – build degraded train features, save, and verify seed non-collision
train_seeds = set(900000 + i for i in range(len(train_idx)))
eval_seeds = set(s * 100000 + j for s in [0, 1, 2] for j in range(len(test_idx)))

print(f'Train seeds range: [{min(train_seeds)}, {max(train_seeds)}]')
print(f'Eval seeds range : [{min(eval_seeds)}, {max(eval_seeds)}]')

assert len(train_seeds & eval_seeds) == 0, 'Collision detected between training seeds and evaluation seeds!'
assert min(train_seeds) >= 900000, f'Minimum train seed {min(train_seeds)} < 900000'
assert max(eval_seeds) < 300000, f'Maximum eval seed {max(eval_seeds)} >= 300000'
print('Seed disjointness assertion passed (no overlap between train augmentation and eval) ✓')

train_deg_feats = []
t0 = time.perf_counter()

for row_index, idx in enumerate(tqdm(train_idx, desc='Extracting degraded train features', mininterval=5.0)):
    path = REPO_ROOT / labels.loc[idx, 'path']
    y = load_fixed(path)
    rng = np.random.default_rng(900000 + row_index)
    yd = degrade(y, SR, rng)
    feat = extract_features(yd)
    train_deg_feats.append(feat)

elapsed = time.perf_counter() - t0
print(f'Degraded train feature extraction completed in {elapsed:.2f} s ({elapsed/60:.2f} min)')

X_train_deg = np.array(train_deg_feats, dtype=np.float32)
assert X_train_deg.shape == (5753, 128), f'Expected shape (5753, 128), got {X_train_deg.shape}'
assert X_train_deg.dtype == np.float32, f'Expected dtype float32, got {X_train_deg.dtype}'
assert not np.isnan(X_train_deg).any(), 'NaN values found in X_train_deg'
assert not np.isinf(X_train_deg).any(), 'Inf values found in X_train_deg'

out_deg_train_path = REPO_ROOT / 'data/processed/audio_features_train_degraded.npy'
np.save(out_deg_train_path, X_train_deg)
print(f'Saved audio_features_train_degraded.npy to: {out_deg_train_path}')
print('Cell 3 asserts passed ✓')

Train seeds range: [900000, 905752]
Eval seeds range : [0, 200604]
Seed disjointness assertion passed (no overlap between train augmentation and eval) ✓


Extracting degraded train features:   0%|          | 0/5753 [00:00<?, ?it/s]

Extracting degraded train features:   0%|          | 1/5753 [00:06<10:50:43,  6.79s/it]

Extracting degraded train features:   2%|▏         | 106/5753 [00:11<08:41, 10.82it/s] 

Extracting degraded train features:   3%|▎         | 180/5753 [00:16<07:26, 12.47it/s]

Extracting degraded train features:   5%|▍         | 264/5753 [00:21<06:30, 14.07it/s]

... (49 outputs left out)

Extracting degraded train features:  97%|█████████▋| 5564/5753 [04:39<00:08, 21.82it/s]

Extracting degraded train features:  99%|█████████▊| 5674/5753 [04:44<00:03, 21.62it/s]

Extracting degraded train features: 100%|██████████| 5753/5753 [04:48<00:00, 19.95it/s]

Degraded train feature extraction completed in 288.40 s (4.81 min)
Saved audio_features_train_degraded.npy to: C:\Users\LENOVO\Desktop\CityPulse\data\processed\audio_features_train_degraded.npy
Cell 3 asserts passed ✓


In [4]:
# Cell 4 – build degraded test features for fold 10 (seeds 0, 1, 2) and verify consistency
X_test_deg = {}

for seed in [0, 1, 2]:
    feats_seed = []
    for row_index, idx in enumerate(tqdm(test_idx, desc=f'Degrading fold 10 (seed {seed})', mininterval=2.0)):
        path = REPO_ROOT / labels.loc[idx, 'path']
        y = load_fixed(path)
        rng = np.random.default_rng(seed * 100000 + row_index)
        yd = degrade(y, SR, rng)
        feature = extract_features(yd)
        feats_seed.append(feature)
    X_test_deg[seed] = np.array(feats_seed, dtype=np.float32)
    assert X_test_deg[seed].shape == (605, 128)

# Check two seeds (0 and 1) against recomputation of 20 random rows
sample_20_rows = np.random.default_rng(42).choice(len(test_idx), 20, replace=False)
print(f'Verifying consistency on 20 random rows: {sorted(sample_20_rows)}')

for s in [0, 1]:
    for r_idx in sample_20_rows:
        orig_idx = test_idx[r_idx]
        path = REPO_ROOT / labels.loc[orig_idx, 'path']
        y_check = load_fixed(path)
        yd_check = degrade(y_check, SR, np.random.default_rng(s * 100000 + r_idx))
        feat_check = extract_features(yd_check)
        
        diff = float(np.max(np.abs(feat_check - X_test_deg[s][r_idx])))
        assert np.allclose(feat_check, X_test_deg[s][r_idx], atol=1e-6), f'Seed {s}, row {r_idx} mismatch: diff {diff}'

print('Recomputed 20 random rows for seeds 0 and 1 match X_test_deg exactly (allclose atol=1e-6) ✓')
print('Cell 4 asserts passed ✓')

Degrading fold 10 (seed 0):   0%|          | 0/605 [00:00<?, ?it/s]

Degrading fold 10 (seed 0):  10%|█         | 63/605 [00:02<00:17, 31.17it/s]

Degrading fold 10 (seed 0):  21%|██▏       | 129/605 [00:04<00:14, 32.21it/s]

Degrading fold 10 (seed 0):  32%|███▏      | 194/605 [00:07<00:16, 25.52it/s]

Degrading fold 10 (seed 0):  41%|████      | 248/605 [00:09<00:13, 26.01it/s]

Degrading fold 10 (seed 0):  50%|█████     | 305/605 [00:11<00:11, 26.73it/s]

... (25 outputs left out)

Degrading fold 10 (seed 2): 100%|█████████▉| 604/605 [00:20<00:00, 31.86it/s]

Degrading fold 10 (seed 2): 100%|██████████| 605/605 [00:20<00:00, 28.88it/s]

Verifying consistency on 20 random rows: [np.int64(50), np.int64(52), np.int64(56), np.int64(77), np.int64(119), np.int64(255), np.int64(258), np.int64(272), np.int64(308), np.int64(313), np.int64(384), np.int64(413), np.int64(430), np.int64(439), np.int64(454), np.int64(455), np.int64(472), np.int64(507), np.int64(582), np.int64(603)]


Recomputed 20 random rows for seeds 0 and 1 match X_test_deg exactly (allclose atol=1e-6) ✓
Cell 4 asserts passed ✓


In [5]:
# Cell 5 – train four models: SVM-clean, SVM-mixed, RF-clean, RF-mixed
X_train_mixed = np.vstack([X_train_clean, X_train_deg])
y_train_mixed = np.concatenate([y_train, y_train])

print(f'X_train_clean shape : {X_train_clean.shape}, y_train length: {len(y_train)}')
print(f'X_train_mixed shape : {X_train_mixed.shape}, y_train_mixed length: {len(y_train_mixed)}')

assert X_train_mixed.shape == (11506, 128), f'Expected mixed train shape (11506, 128), got {X_train_mixed.shape}'
assert len(y_train_mixed) == 11506, f'Expected 11506 mixed labels, got {len(y_train_mixed)}'

# Define fixed architectures
def build_svm():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(C=10, gamma='scale', probability=True, random_state=42))
    ])

def build_rf():
    return RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')

models_dict = {}
fit_times = {}

# 1. SVM-clean
print('\nFitting SVM-clean on clean train data (5,753 samples)...')
svm_clean = build_svm()
t0 = time.perf_counter()
svm_clean.fit(X_train_clean, y_train)
fit_times['SVM-clean'] = time.perf_counter() - t0
models_dict['SVM-clean'] = svm_clean
print(f'SVM-clean fit time: {fit_times["SVM-clean"]:.2f} s')

# 2. SVM-mixed
print('Fitting SVM-mixed on clean + degraded train data (11,506 samples)...')
svm_mixed = build_svm()
t0 = time.perf_counter()
svm_mixed.fit(X_train_mixed, y_train_mixed)
fit_times['SVM-mixed'] = time.perf_counter() - t0
models_dict['SVM-mixed'] = svm_mixed
print(f'SVM-mixed fit time: {fit_times["SVM-mixed"]:.2f} s')

# 3. RF-clean
print('Fitting RF-clean on clean train data (5,753 samples)...')
rf_clean = build_rf()
t0 = time.perf_counter()
rf_clean.fit(X_train_clean, y_train)
fit_times['RF-clean'] = time.perf_counter() - t0
models_dict['RF-clean'] = rf_clean
print(f'RF-clean fit time: {fit_times["RF-clean"]:.2f} s')

# 4. RF-mixed
print('Fitting RF-mixed on clean + degraded train data (11,506 samples)...')
rf_mixed = build_rf()
t0 = time.perf_counter()
rf_mixed.fit(X_train_mixed, y_train_mixed)
fit_times['RF-mixed'] = time.perf_counter() - t0
models_dict['RF-mixed'] = rf_mixed
print(f'RF-mixed fit time: {fit_times["RF-mixed"]:.2f} s')

print('\nAll 4 models successfully trained ✓')
print('Cell 5 completed ✓')

X_train_clean shape : (5753, 128), y_train length: 5753
X_train_mixed shape : (11506, 128), y_train_mixed length: 11506

Fitting SVM-clean on clean train data (5,753 samples)...


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM-clean fit time: 11.50 s
Fitting SVM-mixed on clean + degraded train data (11,506 samples)...


C:\Users\LENOVO\miniconda3\envs\citypulse\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM-mixed fit time: 88.51 s
Fitting RF-clean on clean train data (5,753 samples)...


RF-clean fit time: 5.30 s
Fitting RF-mixed on clean + degraded train data (11,506 samples)...


RF-mixed fit time: 11.04 s

All 4 models successfully trained ✓
Cell 5 completed ✓


In [6]:
# Cell 6 – evaluate on fold-10 clean and fold-10 degraded (mean and std over 3 seeds)
eval_results = []
preds_all = {}

for m_name, model in models_dict.items():
    preds_all[m_name] = {}
    
    # Clean test
    y_pred_clean = model.predict(X_test_clean)
    preds_all[m_name]['clean'] = y_pred_clean
    acc_clean = float(accuracy_score(y_test, y_pred_clean))
    mf1_clean = float(f1_score(y_test, y_pred_clean, average='macro'))
    
    # Degraded test across 3 seeds
    deg_accs = []
    deg_mf1s = []
    for s in [0, 1, 2]:
        y_pred_deg = model.predict(X_test_deg[s])
        preds_all[m_name][f'deg_seed{s}'] = y_pred_deg
        deg_accs.append(float(accuracy_score(y_test, y_pred_deg)))
        deg_mf1s.append(float(f1_score(y_test, y_pred_deg, average='macro')))
        
    deg_acc_mean = float(np.mean(deg_accs))
    deg_acc_std = float(np.std(deg_accs))
    deg_mf1_mean = float(np.mean(deg_mf1s))
    deg_mf1_std = float(np.std(deg_mf1s))
    
    eval_results.append({
        'model': m_name,
        'clean_acc': acc_clean,
        'clean_macro_f1': mf1_clean,
        'deg_acc_mean': deg_acc_mean,
        'deg_acc_std': deg_acc_std,
        'deg_macro_f1_mean': deg_mf1_mean,
        'deg_macro_f1_std': deg_mf1_std
    })

df_eval = pd.DataFrame(eval_results).set_index('model')
print('=== Clean vs Degraded Evaluation Summary ===')
print(df_eval.round(4).to_string())

# Print accuracy changes: mixed minus clean-trained
print('\n=== Accuracy Changes: Mixed minus Clean-trained ===')
for family in ['SVM', 'RF']:
    clean_m = df_eval.loc[f'{family}-clean']
    mixed_m = df_eval.loc[f'{family}-mixed']
    
    delta_clean_acc = mixed_m['clean_acc'] - clean_m['clean_acc']
    delta_clean_pts = delta_clean_acc * 100.0
    delta_deg_acc = mixed_m['deg_acc_mean'] - clean_m['deg_acc_mean']
    delta_deg_pts = delta_deg_acc * 100.0
    
    print(f'{family} Family:')
    print(f'  On Clean Test   : {clean_m["clean_acc"]*100:.2f}% -> {mixed_m["clean_acc"]*100:.2f}% (change: {delta_clean_pts:+.2f} percentage points)')
    print(f'  On Degraded Test: {clean_m["deg_acc_mean"]*100:.2f}% -> {mixed_m["deg_acc_mean"]*100:.2f}% (change: {delta_deg_pts:+.2f} percentage points)\n')

print('Cell 6 completed ✓')

=== Clean vs Degraded Evaluation Summary ===
           clean_acc  clean_macro_f1  deg_acc_mean  deg_acc_std  deg_macro_f1_mean  deg_macro_f1_std
model                                                                                               
SVM-clean     0.7967          0.7780        0.5168       0.0090             0.4822            0.0113
SVM-mixed     0.7818          0.7656        0.7526       0.0008             0.7409            0.0014
RF-clean      0.7521          0.7178        0.4661       0.0129             0.4137            0.0104
RF-mixed      0.7719          0.7441        0.7669       0.0040             0.7452            0.0037

=== Accuracy Changes: Mixed minus Clean-trained ===
SVM Family:
  On Clean Test   : 79.67% -> 78.18% (change: -1.49 percentage points)
  On Degraded Test: 51.68% -> 75.26% (change: +23.58 percentage points)

RF Family:
  On Clean Test   : 75.21% -> 77.19% (change: +1.98 percentage points)
  On Degraded Test: 46.61% -> 76.69% (change: +30.08 perce

In [7]:
# Cell 7 – per-class F1 for mixed models, confusion matrix plot, model saving, and sha256
classes = sorted(np.unique(y_test))

# Per-class F1 for mixed models on clean and degraded test
per_class_mixed = []
for m_name in ['SVM-mixed', 'RF-mixed']:
    clean_rep = classification_report(y_test, preds_all[m_name]['clean'], output_dict=True)
    
    deg_f1s_seeds = {cls: [] for cls in classes}
    for s in [0, 1, 2]:
        rep_s = classification_report(y_test, preds_all[m_name][f'deg_seed{s}'], output_dict=True)
        for cls in classes:
            deg_f1s_seeds[cls].append(rep_s[cls]['f1-score'])
            
    for cls in classes:
        per_class_mixed.append({
            'model': m_name,
            'app_class': cls,
            'clean_f1': clean_rep[cls]['f1-score'],
            'degraded_mean_f1': float(np.mean(deg_f1s_seeds[cls]))
        })

df_per_class_mixed = pd.DataFrame(per_class_mixed)
print('=== Per-class F1 for Mixed Models (Clean vs Degraded Mean) ===')
print(df_per_class_mixed.round(4).to_string(index=False))

# Confusion Matrix for SVM-mixed on seed 0 degraded test
cm_svm_mixed_deg0 = confusion_matrix(y_test, preds_all['SVM-mixed']['deg_seed0'], labels=classes)
cm_df = pd.DataFrame(cm_svm_mixed_deg0, index=classes, columns=classes)
cm_df.index.name = 'True'
cm_df.columns.name = 'Predicted'

print('\n=== SVM-mixed Confusion Matrix (Degraded Seed 0) ===')
print(cm_df.to_string())

fig_dir = REPO_ROOT / 'reports/figures'
fig_dir.mkdir(parents=True, exist_ok=True)
cm_fig_path = fig_dir / 'svm_mixed_confusion_degraded.png'

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_svm_mixed_deg0, display_labels=classes)
disp.plot(ax=ax, cmap='Blues', colorbar=True)
plt.title('SVM-mixed - Confusion Matrix (Degraded Seed 0, Test Fold 10)')
plt.tight_layout()
plt.savefig(cm_fig_path, dpi=150)
plt.close(fig)
print(f'\nSaved SVM-mixed degraded confusion matrix plot to: {cm_fig_path}')

# Save mixed candidate models
models_dir = REPO_ROOT / 'models'
models_dir.mkdir(parents=True, exist_ok=True)
svm_mixed_path = models_dir / 'svm_mixed_candidate.joblib'
rf_mixed_path = models_dir / 'rf_mixed_candidate.joblib'
joblib.dump(models_dict['SVM-mixed'], svm_mixed_path)
joblib.dump(models_dict['RF-mixed'], rf_mixed_path)
print(f'Saved models/svm_mixed_candidate.joblib')
print(f'Saved models/rf_mixed_candidate.joblib')

# Save reports/degraded_training_eval.csv
csv_rows = []
for m_name in ['SVM-clean', 'SVM-mixed', 'RF-clean', 'RF-mixed']:
    for c_name in ['clean', 'deg_seed0', 'deg_seed1', 'deg_seed2']:
        y_p = preds_all[m_name][c_name]
        acc_val = float(accuracy_score(y_test, y_p))
        mf1_val = float(f1_score(y_test, y_p, average='macro'))
        rep = classification_report(y_test, y_p, output_dict=True)
        
        r_dict = {
            'model': m_name,
            'condition': c_name,
            'accuracy': acc_val,
            'macro_f1': mf1_val
        }
        for cls in classes:
            r_dict[f'f1_{cls}'] = rep[cls]['f1-score']
        csv_rows.append(r_dict)

csv_path = REPO_ROOT / 'reports/degraded_training_eval.csv'
df_export = pd.DataFrame(csv_rows)
df_export.to_csv(csv_path, index=False)
print(f'Saved reports/degraded_training_eval.csv')

# Print sha256 of all new files
new_files = [
    REPO_ROOT / 'data/processed/audio_features_train_degraded.npy',
    csv_path,
    cm_fig_path,
    svm_mixed_path,
    rf_mixed_path
]

print('\n=== SHA256 Hashes of All New Files ===')
for f in new_files:
    sha = compute_sha256(f)
    rel = f.relative_to(REPO_ROOT)
    print(f'{str(rel):45s}: {sha}')

print('Cell 7 completed ✓')

=== Per-class F1 for Mixed Models (Clean vs Degraded Mean) ===
    model         app_class  clean_f1  degraded_mean_f1
SVM-mixed      construction    0.8241            0.7906
SVM-mixed              hvac    0.6820            0.6732
SVM-mixed music_loudspeaker    0.8286            0.7780
SVM-mixed             siren    0.6531            0.6471
SVM-mixed      traffic_horn    0.8403            0.8155
 RF-mixed      construction    0.8516            0.8290
 RF-mixed              hvac    0.7487            0.7705
 RF-mixed music_loudspeaker    0.7619            0.7478
 RF-mixed             siren    0.5753            0.5928
 RF-mixed      traffic_horn    0.7830            0.7857

=== SVM-mixed Confusion Matrix (Degraded Seed 0) ===
Predicted          construction  hvac  music_loudspeaker  siren  traffic_horn
True                                                                         
construction                151    25                  8      8             4
hvac                         14  


Saved SVM-mixed degraded confusion matrix plot to: C:\Users\LENOVO\Desktop\CityPulse\reports\figures\svm_mixed_confusion_degraded.png


Saved models/svm_mixed_candidate.joblib
Saved models/rf_mixed_candidate.joblib


Saved reports/degraded_training_eval.csv

=== SHA256 Hashes of All New Files ===
data\processed\audio_features_train_degraded.npy: 99729cd0f0f57057b4de93e2daff3852b04b666d1d0cc1171cc392900a993d7f
reports\degraded_training_eval.csv           : 87013db0e273e7f6a3e6621451dbb2268d5efd2bee84329930dd225a14009fb9
reports\figures\svm_mixed_confusion_degraded.png: b7d1d8baf3b8ee1177b21db14fd485ca942ca25f673542d40d9dea2bc9abf9d8
models\svm_mixed_candidate.joblib            : 72543a986a3178f952f7a935ab7054ad44cfc1e52dbcae30a2756fee95dbb03d


models\rf_mixed_candidate.joblib             : 7e2ecaefce463234cc71ad43e73a7d9cf0e73ee78e35ccde43317b6ffaa15a56
Cell 7 completed ✓


In [8]:
# Cell 8 – sanity check: assert SVM-clean fold-10 clean accuracy equals 0.7967 to 3 decimals
svm_clean_acc = df_eval.loc['SVM-clean', 'clean_acc']
print(f'SVM-clean fold-10 clean accuracy: {svm_clean_acc:.4f}')

assert round(svm_clean_acc, 3) == 0.797, f'Sanity check failed: expected 0.797 (from 0.7967), got {svm_clean_acc:.4f}'
print('Cell 8 sanity assert passed: SVM-clean matches baseline 0.7967 to 3 decimals ✓')

SVM-clean fold-10 clean accuracy: 0.7967
Cell 8 sanity assert passed: SVM-clean matches baseline 0.7967 to 3 decimals ✓
